In [0]:
create schema if not exists nyc_taxi.gold;

In [0]:
%python
silver_df=spark.table("nyc_taxi.silver.yellow_trip_clean")
silver_df.count()

In [0]:
%python
from pyspark.sql.functions import count, sum

daily_revenue_df = (
    silver_df
    .groupBy("trip_date")
    .agg(
        count("*").alias("total_trips"),
        sum("total_amount").alias("revenue")
    )
)

display(daily_revenue_df)

In [0]:
%python
daily_revenue_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nyc_taxi.gold.daily_revenue")

In [0]:
%python
from pyspark.sql.functions import count

trips_by_hour_df = (
    silver_df
    .groupBy("pickup_hour")
    .agg(count("*").alias("trip_count"))
)

display(trips_by_hour_df)

In [0]:
%python
trips_by_hour_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nyc_taxi.gold.trips_by_hour")

In [0]:
%python
from pyspark.sql.functions import sum, count

payment_analysis = (
    silver_df
    .groupBy("payment_type")
    .agg(
        count("*").alias("trip_count"),
        sum("total_amount").alias("revenue")
    )
)

payment_analysis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nyc_taxi.gold.payment_analysis")

In [0]:
%python
from pyspark.sql.functions import count

top_pickups = (
    silver_df
    .groupBy("PULocationID")
    .agg(count("*").alias("trip_count"))
    .orderBy("trip_count", ascending=False)
)

top_pickups.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nyc_taxi.gold.top_pickup_locations")

In [0]:
%python
from pyspark.sql.functions import count

top_dropoff_df = (
    silver_df
    .groupBy("DOLocationID")
    .agg(count("*").alias("trip_count"))
    .orderBy("trip_count", ascending=False)
)

display(top_dropoff_df)

In [0]:
%python


In [0]:
%python
top_dropoff_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("nyc_taxi.gold.top_dropoff_locations")

In [0]:
%python
payment_revenue_df = (
    silver_df
    .groupBy("payment_type")
    .agg(
        count("*").alias("trip_count"),
        sum("total_amount").alias("revenue")
    )
)

In [0]:
%python
payment_revenue_df.write.format("delta").mode("overwrite").saveAsTable("nyc_taxi.gold.payment_revenue")

In [0]:
%python
from pyspark.sql.functions import avg

duration_by_hour_df = (
    silver_df
    .groupBy("pickup_hour")
    .agg(
        avg("trip_duration_minutes").alias("avg_duration")
    )
)

In [0]:
%python
duration_by_hour_df.write.format("delta").mode("overwrite").saveAsTable("nyc_taxi.gold.duration_by_hour")
       

In [0]:
%python
location_revenue_df = (
    silver_df
    .groupBy("PULocationID")
    .agg(
        sum("total_amount").alias("revenue"),
        count("*").alias("trip_count")
    )
)
display(location_revenue_df)

In [0]:
%python
location_revenue_df.write.format("delta").mode("overwrite").saveAsTable("nyc_taxi.gold.location_revenue")

In [0]:
%python
from pyspark.sql.functions import avg, sum, count

daily_kpi_df = (
    silver_df
    .groupBy("trip_date")
    .agg(
        count("*").alias("total_trips"),
        sum("total_amount").alias("total_revenue"),
        avg("trip_distance").alias("avg_distance"),
        avg("trip_duration_minutes").alias("avg_duration")
    )
)
display(daily_kpi_df)


In [0]:
%python
from pyspark.sql.functions import date_format, sum, avg, round

monthly_revenue = (
    silver_df
    .groupBy(
        date_format("tpep_pickup_datetime", "yyyy-MM").alias("month")
    )
    .agg(
        round(sum("total_amount"), 2).alias("total_revenue"),
        round(avg("total_amount"), 2).alias("avg_trip_revenue")
    )
    .orderBy("month")
)

display(monthly_revenue)

In [0]:
%python
from pyspark.sql.functions import date_format

monthly_kpi = (
    silver_df
    .groupBy(
        date_format("trip_date", "yyyy-MM").alias("month")
    )
    .agg(
        count("*").alias("total_trips"),
        sum("total_amount").alias("total_revenue")
    )
)

monthly_kpi.write.mode("overwrite").saveAsTable(
    "nyc_taxi.gold.monthly_kpi"
)

In [0]:
%python
from pyspark.sql.functions import sum, date_format

revenue_by_day = (
    silver_df
    .groupBy(
        date_format("trip_date", "yyyy-MM").alias("month"),
        "day_of_week"
    )
    .agg(
        sum("total_amount").alias("total_revenue")
    )
)

revenue_by_day.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("nyc_taxi.gold.revenue_by_day")

In [0]:
%python
from pyspark.sql.functions import count
from pyspark.sql.functions import count, date_format

trips_by_day = (
    silver_df
    .groupBy(
        date_format("trip_date","yyyy-MM").alias("month"),
        "day_of_week"
    )
    .agg(
        count("*").alias("total_trips")
    )
)

trips_by_day.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("nyc_taxi.gold.trips_by_day")

In [0]:
%python
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

window = Window.orderBy("month")

monthly_growth = (
    monthly_revenue
    .withColumn(
        "previous_month_revenue",
        lag("total_revenue").over(window)
    )
    .withColumn(
        "revenue_growth_percent",
        round(
            (
                (monthly_revenue.total_revenue -
                 lag("total_revenue").over(window))
                /
                lag("total_revenue").over(window)
            )*100,
            2
        )
    )
)

display(monthly_growth)

In [0]:
%python
from pyspark.sql.functions import (
    date_format,
    count,
    sum,
    avg,
    round
)

gold_monthly = (
    silver_df
    .groupBy(
        date_format("tpep_pickup_datetime", "yyyy-MM").alias("month")
    )
    .agg(
        count("*").alias("total_trips"),
        round(sum("total_amount"), 2).alias("total_revenue"),
        round(avg("total_amount"), 2).alias("avg_revenue_per_trip"),
        round(avg("trip_distance"), 2).alias("avg_trip_distance"),
        round(sum("passenger_count"), 0).alias("total_passengers")
    )
    .orderBy("month")
)

display(gold_monthly)
gold_monthly.write.mode("overwrite").saveAsTable(
    "nyc_taxi.gold.monthly_analytics"
)

In [0]:
%python
daily_kpi_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nyc_taxi.gold.daily_kpi")